In [8]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

file_path = '../data/ed2022_sas.sas7bdat'
df = pd.read_sas(file_path)
df.head()

,VMONTH,VDAYR,ARRTIME,WAITTIME,LOV,AGE,AGER,AGEDAYS,RESIDNCE,SEX,...,RX30V3C2,RX30V3C3,RX30V3C4,SETTYPE,YEAR,CSTRATM,CPSUM,PATWT,EDWT,BOARDED
0,9.0,2.0,b'0604',10.0,228.0,23.0,2.0,-7.0,1.0,1.0,...,NaN,NaN,NaN,3.0,2022.0,20122201.0,100001.0,3665.56954,8.36413,-7.0
1,9.0,2.0,b'1053',40.0,319.0,15.0,2.0,-7.0,1.0,2.0,...,NaN,NaN,NaN,3.0,2022.0,20122201.0,100001.0,3665.56954,NaN,-7.0
2,9.0,2.0,b'1419',70.0,551.0,19.0,2.0,-7.0,1.0,2.0,...,NaN,NaN,NaN,3.0,2022.0,20122201.0,100001.0,3665.56954,NaN,-7.0
3,9.0,2.0,b'1825',-7.0,-9.0,0.0,1.0,298.0,1.0,2.0,...,NaN,NaN,NaN,3.0,2022.0,20122201.0,100001.0,3665.56954,NaN,-7.0
4,9.0,2.0,b'2243',14.0,168.0,18.0,2.0,-7.0,1.0,1.0,...,NaN,NaN,NaN,3.0,2022.0,20122201.0,100001.0,3665.56954,NaN,-7.0


In [9]:
# Dataset Overview
print("Dataset Shape:", df.shape)
print("\nColumn Names and Data Types:")
print(df.dtypes)
print("\nMissing Values:")
print(df.isnull().sum())

Dataset Shape: (16025, 913)

Column Names and Data Types:
VMONTH      float64
VDAYR       float64
ARRTIME      object
WAITTIME    float64
LOV         float64
             ...   
CSTRATM     float64
CPSUM       float64
PATWT       float64
EDWT        float64
BOARDED     float64
Length: 913, dtype: object

Missing Values:
VMONTH          0
VDAYR           0
ARRTIME         0
WAITTIME        0
LOV             0
            ...  
CSTRATM         0
CPSUM           0
PATWT           0
EDWT        15837
BOARDED         0
Length: 913, dtype: int64


In [7]:
# Extract column descriptions from NHAMCS documentation PDF
import pdfplumber
import pandas as pd
import re

pdf_path = '../data/doc21-ed-508.pdf'

# Extract text and tables from PDF
try:
    with pdfplumber.open(pdf_path) as pdf:
        print(f"Total pages in PDF: {len(pdf.pages)}\n")
        
        # Extract all text
        full_text = ""
        for page in pdf.pages:
            full_text += page.extract_text()
        
        # Try to find column/variable definitions
        print("Looking for column definitions in the documentation...\n")
        
        # Search for common patterns in variable documentation
        lines = full_text.split('\n')
        
        # Find variable definitions (usually start with a variable name followed by description)
        variables = {}
        current_var = None
        
        for i, line in enumerate(lines):
            # Look for lines that might contain variable definitions
            if re.match(r'^[A-Z]{2,}', line.strip()) and len(line.strip()) < 50:
                current_var = line.strip()
                if i + 1 < len(lines):
                    variables[current_var] = lines[i + 1].strip()
        
        print(f"Found {len(variables)} potential variables")
        print("\nSample variables and descriptions:")
        for var, desc in list(variables.items())[:20]:
            print(f"{var}: {desc[:70]}...")
        
except Exception as e:
    print(f"Error: {e}")
    import traceback
    traceback.print_exc()

Total pages in PDF: 266

Looking for column definitions in the documentation...

Found 325 potential variables

Sample variables and descriptions:
ABSTRACT: This material provides documentation for users of the Emergency Depart...
SUMMARY OF CHANGES FOR 2021: A. Survey Items...
PATIENT VISIT WEIGHT: Micro-data file users should be fully aware of the importance of the "...
RELIABILITY OF ESTIMATES: Researchers should also be aware of the reliability or unreliability o...
PSU sample design is available (5).: 2. Hospitals...
NONRESPONSE: VARIABLE VARIABLE DESCRIPTION DENOMINATOR...
VARIABLE VARIABLE DESCRIPTION DENOMINATOR: %...
WAITTIME in minutes was made 12.0: RACEUN Unimputed race All visits 21.4...
RACEUN Unimputed race All visits 21.4: ETHUN Unimputed ethnicity All visits 14.4...
ETHUN Unimputed ethnicity All visits 14.4: Was the patient transferred...
AMBTRANSFER care facility? ambulance 17.3: Recoded primary expected...
TEMPF (Fahrenheit) All visits 6.5: Initial vital signs: Heart

In [4]:
# Build a comprehensive data dictionary from PDF and current dataframe columns
column_descriptions = {
    # Temporal Variables
    'VMONTH': 'Month of visit',
    'VDAYR': 'Day of week of visit',
    'ARRTIME': 'Arrival time (formatted)',
    'WAITTIME': 'Time from arrival to seeing provider (minutes)',
    'LOV': 'Length of visit (minutes)',
    'BOARDED': 'Number of hours patient remained in ED after decision to admit',
    
    # Demographics
    'AGE': 'Age of patient in years',
    'AGER': 'Age group',
    'AGEDAYS': 'Age in days for pediatric patients',
    'SEX': 'Sex of patient (1=Male, 2=Female)',
    'ETHUN': 'Ethnicity - unimputed (1=Non-Hispanic, 2=Hispanic)',
    'RACEUN': 'Race - unimputed',
    'RESIDNCE': 'Residence area (1=Metropolitan, 2=Micropolitan, 3=Non-core)',
    
    # Clinical Variables
    'TRIAGE': 'Triage acuity level',
    'PAINSCALE': 'Pain scale score (0-10)',
    'TEMPF': 'Temperature in Fahrenheit',
    'PULSE': 'Heart rate (beats per minute)',
    'RESPR': 'Respiratory rate per minute',
    'BPSYS': 'Blood pressure - Systolic',
    'BPDIAS': 'Blood pressure - Diastolic',
    'POPCT': 'Pulse oximetry (percent)',
    
    # Resource Utilization
    'IMAG': 'Imaging ordered',
    'CT': 'CT scan ordered',
    'MRI': 'MRI ordered',
    'XRAY': 'X-ray ordered',
    'ULTRASOUND': 'Ultrasound ordered',
    'LABTEST': 'Laboratory test ordered',
    'MEDORTO': 'Medication ordered/provided at ED',
    
    # Hospital Operations
    'SETTYPE': 'Type of facility (hospital/freestanding)',
    'YEAR': 'Year of data collection',
    'EDWT': 'Probability weight for ED',
    'PATWT': 'Patient sampling weight',
}

# Get all columns from the dataframe
all_columns = df.columns.tolist()

# Create a DataFrame with column information
column_info = []
for col in all_columns:
    description = column_descriptions.get(col, 'See NHAMCS documentation')
    data_type = str(df[col].dtype)
    non_missing = df[col].notna().sum()
    missing_pct = (df[col].isna().sum() / len(df)) * 100
    
    column_info.append({
        'Column_Name': col,
        'Data_Type': data_type,
        'Non_Missing': non_missing,
        'Missing_%': f"{missing_pct:.1f}%",
        'Description': description
    })

column_dict_df = pd.DataFrame(column_info)

print("="*100)
print("COMPLETE DATA DICTIONARY - ED 2022 DATASET (from NHAMCS)")
print("="*100)
print(f"\nTotal columns: {len(column_dict_df)}")
print(f"Total rows: {len(df)}\n")

# Display by category
print("\n" + "="*100)
print("TEMPORAL VARIABLES")
print("="*100)
temporal_display = column_dict_df[column_dict_df['Column_Name'].isin(['VMONTH', 'VDAYR', 'ARRTIME', 'WAITTIME', 'LOV', 'BOARDED'])]
print(temporal_display.to_string(index=False))

print("\n" + "="*100)
print("DEMOGRAPHIC VARIABLES")
print("="*100)
demo_display = column_dict_df[column_dict_df['Column_Name'].isin(['AGE', 'AGER', 'AGEDAYS', 'SEX', 'ETHUN', 'RACEUN', 'RESIDNCE'])]
print(demo_display.to_string(index=False))

print("\n" + "="*100)
print("CLINICAL VARIABLES")
print("="*100)
clinical_display = column_dict_df[column_dict_df['Column_Name'].isin(['TRIAGE', 'PAINSCALE', 'TEMPF', 'PULSE', 'RESPR', 'BPSYS', 'BPDIAS', 'POPCT'])]
print(clinical_display.to_string(index=False))

print("\n" + "="*100)
print("ALL COLUMNS - FULL DATA DICTIONARY")
print("="*100)
print(column_dict_df.to_string(index=False))

COMPLETE DATA DICTIONARY - ED 2022 DATASET (from NHAMCS)

Total columns: 913
Total rows: 16025


TEMPORAL VARIABLES
Column_Name Data_Type  Non_Missing Missing_%                                                    Description
     VMONTH   float64        16025      0.0%                                                 Month of visit
      VDAYR   float64        16025      0.0%                                           Day of week of visit
    ARRTIME    object        16025      0.0%                                       Arrival time (formatted)
   WAITTIME   float64        16025      0.0%                 Time from arrival to seeing provider (minutes)
        LOV   float64        16025      0.0%                                      Length of visit (minutes)
    BOARDED   float64        16025      0.0% Number of hours patient remained in ED after decision to admit

DEMOGRAPHIC VARIABLES
Column_Name Data_Type  Non_Missing Missing_%                                                 Description


In [14]:
# Split columns into thematic dataframes for analysis
from collections import OrderedDict

# Define core groups (explicit + keyword-based fallbacks)
groups = OrderedDict({
    "temporal": {
        "explicit": ["VMONTH", "VDAYR", "ARRTIME", "WAITTIME", "LOV", "BOARDED"],
        "keywords": ["time", "month", "day", "arrival", "wait", "los", "lov", "board", "hour", "minute"]
    },
    "demographics": {
        "explicit": ["AGE", "AGER", "AGEDAYS", "SEX", "ETHUN", "RACEUN", "RESIDNCE"],
        "keywords": ["age", "sex", "gender", "race", "ethnicity", "ethn", "resid", "payer", "insur"]
    },
    "clinical": {
        "explicit": ["TRIAGE", "PAINSCALE", "TEMPF", "PULSE", "RESPR", "BPSYS", "BPDIAS", "POPCT"],
        "keywords": ["triage", "pain", "vital", "temp", "pulse", "resp", "bp", "ox", "diagn", "injur"]
    },
    "resources": {
        "explicit": ["IMAG", "CT", "MRI", "XRAY", "ULTRASOUND", "LABTEST", "MEDORTO"],
        "keywords": ["imag", "ct", "mri", "xray", "ultra", "lab", "test", "med", "drug", "proc"]
    },
    "operations": {
        "explicit": ["SETTYPE", "YEAR", "EDWT", "PATWT"],
        "keywords": ["settype", "edwt", "patwt", "weight", "volume", "bed", "capacity", "fast", "track", "protocol"]
    },
})

# Helper to collect existing columns

def collect_columns(explicit, keywords):
    cols = []
    for c in explicit:
        if c in df.columns:
            cols.append(c)
    for c in df.columns:
        cl = c.lower()
        if any(k in cl for k in keywords):
            cols.append(c)
    seen = set()
    unique_cols = []
    for c in cols:
        if c not in seen:
            seen.add(c)
            unique_cols.append(c)
    return unique_cols

# Build dataframes
df_temporal = df[collect_columns(groups["temporal"]["explicit"], groups["temporal"]["keywords"])].copy()
df_demographics = df[collect_columns(groups["demographics"]["explicit"], groups["demographics"]["keywords"])].copy()
df_clinical = df[collect_columns(groups["clinical"]["explicit"], groups["clinical"]["keywords"])].copy()
df_resources = df[collect_columns(groups["resources"]["explicit"], groups["resources"]["keywords"])].copy()
df_operations = df[collect_columns(groups["operations"]["explicit"], groups["operations"]["keywords"])].copy()

# Summary
summary = pd.DataFrame({
    "dataframe": ["df_temporal", "df_demographics", "df_clinical", "df_resources", "df_operations"],
    "rows": [len(df_temporal), len(df_demographics), len(df_clinical), len(df_resources), len(df_operations)],
    "columns": [df_temporal.shape[1], df_demographics.shape[1], df_clinical.shape[1], df_resources.shape[1], df_operations.shape[1]]
})

print("Created thematic dataframes:")
print(summary.to_string(index=False))

# Show a few columns from each for validation
print("\nSample columns:")
print("Temporal:", list(df_temporal.columns[:12]))
print("Demographics:", list(df_demographics.columns[:12]))
print("Clinical:", list(df_clinical.columns[:12]))
print("Resources:", list(df_resources.columns[:12]))
print("Operations:", list(df_operations.columns[:12]))




Created thematic dataframes:
      dataframe  rows  columns
    df_temporal 16025       11
df_demographics 16025       16
    df_clinical 16025       19
   df_resources 16025      118
  df_operations 16025        9

Sample columns:
Temporal: ['VMONTH', 'VDAYR', 'ARRTIME', 'WAITTIME', 'LOV', 'BOARDED', 'AGEDAYS', 'LOS', 'BOARD', 'BOARDHOS', 'SURGDAY']
Demographics: ['AGE', 'AGER', 'AGEDAYS', 'SEX', 'ETHUN', 'RACEUN', 'RESIDNCE', 'RACER', 'RACERETH', 'ANYIMAGE', 'OTHIMAGE', 'AGEFL']
Clinical: ['PAINSCALE', 'TEMPF', 'PULSE', 'RESPR', 'BPSYS', 'BPDIAS', 'POPCT', 'INJURY', 'INJURY72', 'INJURY_ENC', 'TOXSCREN', 'BPAP']
Resources: ['MRI', 'XRAY', 'POPCT', 'IMMEDR', 'ELECTROL', 'LACTATE', 'HIVTEST', 'FLUTEST', 'PREGTEST', 'OTHRTEST', 'ANYIMAGE', 'CTCONTRAST']
Operations: ['SETTYPE', 'YEAR', 'EDWT', 'PATWT', 'BEDREG', 'IMBED', 'FASTTRAK', 'BEDCZAR', 'BEDDATA']


In [16]:
# Convert ARRTIME to proper time format (HH:MM)
def convert_arrtime_to_time(arr_time):
    """
    Convert ARRTIME from byte format (e.g., b'0604') to time format (HH:MM)
    b'0604' -> '06:04'
    b'1419' -> '14:19'
    """
    try:
        if pd.isna(arr_time):
            return pd.NaT
        
        # Convert bytes to string if needed
        if isinstance(arr_time, bytes):
            time_str = arr_time.decode('utf-8')
        else:
            time_str = str(arr_time)
        
        # Handle cases where time is already in correct format or is missing
        if len(time_str) < 4:
            return pd.NaT
        
        # Extract hours and minutes (last 4 digits)
        hours = time_str[-4:-2]
        minutes = time_str[-2:]
        
        # Create time string in HH:MM format
        time_formatted = f"{hours}:{minutes}"
        
        # Convert to datetime.time object
        return pd.to_datetime(time_formatted, format='%H:%M').time()
    except:
        return pd.NaT

# Apply conversion
df_temporal['ARRTIME'] = df_temporal['ARRTIME'].apply(convert_arrtime_to_time)

print("ARRTIME conversion complete!")
print("\nSample converted ARRTIME values:")
print(df_temporal['ARRTIME'].head(20))
print(f"\nData type: {df_temporal['ARRTIME'].dtype}")
print(f"\nMissing values: {df_temporal['ARRTIME'].isna().sum()}")

ARRTIME conversion complete!

Sample converted ARRTIME values:
0     06:04:00
1     10:53:00
2     14:19:00
3     18:25:00
4     22:43:00
5     09:03:00
6     14:28:00
7     18:30:00
8     05:32:00
9     16:38:00
10    21:48:00
11    10:14:00
12    15:04:00
13    19:03:00
14    12:25:00
15    17:13:00
16    23:09:00
17    08:41:00
18    13:12:00
19    18:13:00
Name: ARRTIME, dtype: object

Data type: object

Missing values: 243
